### Udaplay_01_solution_project — load, process, and format the provided game JSON files.

This solution loads, processes, and formats the games.json file

## 1 Setting up dependencies

import the necessary packages and libraries and load environment

In [10]:
from dotenv import load_dotenv
import os

from tools import (
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
    long_term_memory,
    build_memory_registration_tool,
    build_memory_search_tool,
)

from lib.agents import Agent
from lib.messages import BaseMessage
from typing import List

load_dotenv("config.env")
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

# 1.1 Create printer function

implement message printer private function to better visualize thinking process and output of the agent

In [11]:
def _print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(
            f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})"
        )

# 2 RAG, Tavily Client, Long Term Memory initialization


In [12]:
import json
from lib.tooling import tool
from tavily import TavilyClient
from lib.llm import LLM
from lib.rag import RAG
import os
from lib.vector_db import VectorStoreManager
from lib.documents import Document
from lib.memory import LongTermMemory, MemoryFragment

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

rag_llm = LLM(model="gpt-4o-mini", temperature=0.3)

vector_store_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"))
games_data_vector_store = vector_store_manager.create_store(store_name="games_data", force=True)
long_term_memory = LongTermMemory(vector_store_manager)


@tool
def get_games(num_games:int=1, top:bool=True) -> str:
    """
    Returns the top or bottom N games with highest or lowest scores.    
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    with open('games.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        # Sort the games list by Score
        # If top is True, descending order
        sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)

        # Return the N games
        return str(sorted_games[:num_games])
    
games_data_vector_store.add(Document(content=get_games(num_games=10, top=True)))
games_rag = RAG(llm=rag_llm, vector_store=games_data_vector_store)

# 3 Query the vector database

In [13]:
#get the top 10 games in the vector database
games_str=get_games(num_games=10, top=True)
print("Top 10 games:\n", games_str)

Top 10 games:
 [{'Game': 'The Legend of Zelda: Breath of the Wild', 'Platform': 'Switch', 'Score': 98}, {'Game': 'Super Mario Odyssey', 'Platform': 'Switch', 'Score': 97}, {'Game': 'Metroid Prime', 'Platform': 'GameCube', 'Score': 97}, {'Game': 'Super Smash Bros. Brawl', 'Platform': 'Wii', 'Score': 93}, {'Game': 'Mario Kart 8 Deluxe', 'Platform': 'Switch', 'Score': 92}, {'Game': 'Fire Emblem: Awakening', 'Platform': '3DS', 'Score': 92}, {'Game': 'Animal Crossing: New Leaf', 'Platform': '3DS', 'Score': 88}, {'Game': 'Donkey Kong Country Returns', 'Platform': 'Wii', 'Score': 87}, {'Game': "Luigi's Mansion 3", 'Platform': 'Switch', 'Score': 86}, {'Game': 'Pikmin 3', 'Platform': 'Wii U', 'Score': 85}]
